#1. Gọi thư viện

In [ ]:
!pip install pmdarima

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error
from pmdarima.arima.utils import nsdiffs, ndiffs
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.statespace.sarimax import SARIMAX
import itertools
import warnings
drive.mount('/content/drive')
folder = Path("/content/drive/MyDrive/Tân Cảng Hiệp Phước/Data_Train_Val_Test")


#2. Đọc tập tin

In [ ]:
df_train=pd.read_csv(folder / "Train.csv")
df_val=pd.read_csv(folder / "Validation.csv")
df_test=pd.read_csv(folder / "Test.csv")

#3. Huấn luyện và lựa chọn cấu hình

##3.1 Chuẩn hóa dữ liệu

In [ ]:
y_train = df_train['TEU'].astype(float).reset_index(drop=True)
y_val = df_val['TEU'].astype(float).reset_index(drop=True)
y_test = df_test['TEU'].astype(float).reset_index(drop=True)

# Kiểm tra dữ liệu
assert len(y_train) > 0, "Tập Train đang rỗng!"
assert len(y_val) > 0, "Tập Validation đang rỗng!"
assert len(y_test) > 0, "Tập Test đang rỗng!"

assert not y_train.isna().any(), "Train có giá trị thiếu!"
assert not y_val.isna().any(), "Validation có giá trị thiếu!"
assert not y_test.isna().any(), "Test có giá trị thiếu!"

##3.2 Gộp tập train và validation


In [ ]:
y_train_val = pd.concat(
    [y_train, y_val],
    ignore_index=True
).astype(float)

# Tạo index tháng, bắt đầu từ tháng 01/2021
y_train_val.index = pd.date_range(
    start="2021-01-01",
    periods=len(y_train_val),
    freq="MS"
)

In [ ]:
# Tạo index cho tập Test
y_test = pd.Series(
    y_test.to_numpy(),
    index=pd.date_range(
        start=y_train_val.index[-1] + pd.offsets.MonthBegin(1),
        periods=len(y_test),
        freq="MS"
    ),
    name="Thực tế"
)

print("=== THÔNG TIN DỮ LIỆU ===")
print("Train + Validation:",
      y_train_val.index.min(), "->", y_train_val.index.max())
print("Số quan sát Train + Validation:", len(y_train_val))

print("Test:",
      y_test.index.min(), "->", y_test.index.max())
print("Số quan sát Test:", len(y_test))

##3.3 Thiết lập không gian tham số

In [ ]:
p_values = range(0, 3)
d_values = [0, 1]
q_values = range(0, 3)

P_values = range(0, 3)
D_values = [0, 1]
Q_values = range(0, 3)

m = 12

param_combinations = list(itertools.product(
    p_values, d_values, q_values,
    P_values, D_values, Q_values
))

results = []
errors = []

##3.4 Huấn luyện và lựa chọn 6 cấu hình xếp hạng theo chỉ số AICc

In [ ]:
results = []
errors = []

for p, d, q, P, D, Q in param_combinations:

    order = (p, d, q)
    seasonal_order = (P, D, Q, m)

    try:

        # Fit trên Train
        model = SARIMAX(
            y_train,
            order=order,
            seasonal_order=seasonal_order,
            trend="n",
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted_model = model.fit(disp=False)

        # AIC, BIC
        aic = fitted_model.aic
        bic = fitted_model.bic

        # Số tham số ước lượng
        k = len(fitted_model.params)
        n = fitted_model.nobs

        # AICc
        if n - k - 1 > 0:
            aicc = aic + (
                2 * k * (k + 1)
            ) / (n - k - 1)
        else:
            aicc = np.inf

        # Dự báo trên Validation
        val_pred = np.asarray(
            fitted_model.forecast(steps=len(y_val))
        ).reshape(-1)

        y_val_actual = y_val.to_numpy()

        # Sai số tuyệt đối
        abs_errors = np.abs(y_val_actual - val_pred)

        # MAE
        mae = np.mean(abs_errors)

        # STD MAE
        std_mae = (
            np.std(abs_errors, ddof=1)
            if len(abs_errors) > 1
            else np.nan
        )

        # Lưu kết quả
        if np.isfinite(aicc):

            results.append({
                "order": order,
                "seasonal_order": seasonal_order,
                "AIC": aic,
                "AICc": aicc,
                "BIC": bic,
                "MAE": mae,
                "STD MAE": std_mae
            })

    except Exception as e:

        errors.append({
            "order": order,
            "seasonal_order": seasonal_order,
            "Error": str(e)
        })

##3.5 Tổng hợp và đánh giá 6 cấu hình

In [ ]:
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("Không có mô hình SARIMA nào ước lượng thành công.")

results_df = results_df.sort_values(
    by="AICc",
    ascending=True
).reset_index(drop=True)

# Hiển thị 10 mô hình có AICc thấp nhất
print("\n=== 10 MÔ HÌNH SARIMA CÓ AICc NHỎ NHẤT ===")

display_cols = [
    "order",
    "seasonal_order",
    "AIC",
    "AICc",
    "BIC",
    "MAE",
    "STD MAE"
]

print(
    results_df[display_cols]
    .head(10)
    .round(4)
    .to_string(index=False)
)

##3.6 Huấn luyện lại dữ liệu trên tập train và validaiton

In [ ]:
best_model = SARIMAX(
    y_train_val,
    order=best_order,
    seasonal_order=best_seasonal_order,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

#4. Dự báo và trực quan hóa kết quả trên tập test

##4.1 Dự báo kết quả trên tập test

In [ ]:
forecast_result = best_model.get_forecast(
    steps=len(y_test)
)
y_pred = pd.Series(
    np.asarray(forecast_result.predicted_mean).reshape(-1),
    index=y_test.index,
    name="Dự báo"
)

##4.2 Trực quan hóa kết quả

In [ ]:
#1. Tính toán các chỉ số
# Giá trị thực tế
y_thucte = y_test.to_numpy()

# Giá trị dự báo
y_dudoan = y_pred.to_numpy()

# Sai số tuyệt đối
errors_abs = np.abs(
    y_thucte - y_dudoan
)

# MAE
mae = np.mean(errors_abs)

# Độ lệch chuẩn mẫu của sai số tuyệt đối
std_mae = (
    np.std(errors_abs, ddof=1)
    if len(errors_abs) > 1
    else np.nan
)

# MAPE: loại bỏ các giá trị thực tế bằng 0
non_zero_mask = y_thucte != 0

mape = (
    np.mean(
        errors_abs[non_zero_mask] /
        np.abs(y_thucte[non_zero_mask])
    ) * 100
    if non_zero_mask.any()
    else np.nan
)


# 2. Bảng so sánh kết quả thực tế và dự báo
comparison = pd.DataFrame({

    "Y thực tế": y_thucte,

    "Y dự đoán": y_dudoan,

    "Sai số tuyệt đối": errors_abs,

    "Phần trăm sai số (%)": np.where(
        y_thucte != 0,
        errors_abs / np.abs(y_thucte) * 100,
        np.nan
    )})

print("KẾT QUẢ DỰ BÁO TRÊN TẬP TEST")
print(comparison.round(2).to_string(index=False)
)

#3. Vẽ biểu đồ
plt.figure(figsize=(14, 6))

# Train + Validation
plt.plot(
    y_train_val.index,
    y_train_val.values,
    label="Train + Validation",
    marker="o"
)

# Giá trị thực tế Test
plt.plot(
    y_test.index,
    y_test.values,
    label="Thực tế (Test)",
    color="black",
    marker="o"
)

# Giá trị dự báo
plt.plot(
    y_pred.index,
    y_pred.values,
    label="Dự báo SARIMA",
    color="red",
    linestyle="--",
    marker="o"
)

# Đường phân chia Test
plt.axvline(
    x=y_test.index[0],
    color="gray",
    linestyle=":",
    label="Bắt đầu tập Test"
)

plt.title(
    f"DỰ BÁO SẢN LƯỢNG TEU BẰNG SARIMA\n"
    f"order={best_order}, "
    f"seasonal_order={best_seasonal_order}",
    fontsize=13,
    fontweight="bold"
)

plt.xlabel("Tháng")
plt.ylabel("Sản lượng TEU")

plt.xticks(rotation=45)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


#5. Lưu mô hình

In [ ]:
# Thư mục lưu mô hình
folder = Path("/content/drive/MyDrive/Tân Cảng Hiệp Phước/Train - Model Selection")
folder.mkdir(parents=True, exist_ok=True)

# Đường dẫn lưu mô hình
model_path = folder / "best_model_sa.pkl"

# Lưu mô hình
with open(model_path, "wb") as file:
    pickle.dump(best_model, file)

print(f"Đã lưu mô hình tại: {model_path}")